# Build 3 — Unity AI Gateway: inference-table & guardrail validation

This notebook is the **execution proof** for Build 3. Running it against the live workspace shows that:
1. the inference-log **catalog/schema + both auto-capture inference tables** were created and are capturing (not just declared as code);
2. **[bonus]** the coding agent's endpoint is **NOT bound by the app's `block_all_lakebase_data` guardrail** — the same all-data prompt succeeds on the coding agent's table but is blocked on the app's.

Everything below runs read-only SQL against Unity Catalog + `system.ai_gateway.usage` (no AI Gateway model calls, so the $0.05 budget block does not affect it). Export this notebook **with cell outputs** as the committed evidence.


In [1]:
CAT = "brightwave_techsummit27_catalog"
WS  = "7474647541759877"
APP_TABLE   = f"{CAT}.inference_logs.campaign_desk_llm_payload"   # app endpoint  (guardrail: block_all_lakebase_data)
CODEX_TABLE = f"{CAT}.inference_logs.codex_agent_payload"         # coding-agent endpoint (NO guardrail)
print("App endpoint inference table  :", APP_TABLE)
print("Coding-agent inference table  :", CODEX_TABLE)


App endpoint inference table  : brightwave_techsummit27_catalog.inference_logs.campaign_desk_llm_payload
Coding-agent inference table  : brightwave_techsummit27_catalog.inference_logs.codex_agent_payload


## 1. Catalog + inference tables created by committed code — they EXIST
(`serving_endpoint_spec.json` / `create_serving_endpoints.sh` declared them; this proves the gateway auto-created them.)


In [2]:
display(spark.sql(f"SHOW TABLES IN {CAT}.inference_logs"))


database        tableName                  isTemporary
--------------  -------------------------  -----------
inference_logs  campaign_desk_llm_payload  False      
inference_logs  codex_agent_payload        False      
inference_logs  coding_agent_payload       False      
[3 row(s)]


## 2. Both auto-capture tables are actively CAPTURING (row counts > 0)


In [3]:
display(spark.sql(f"""
  SELECT 'campaign_desk_llm_payload (app)'      AS inference_table, COUNT(*) AS captured_rows FROM {APP_TABLE}
  UNION ALL
  SELECT 'codex_agent_payload (coding agent)'   AS inference_table, COUNT(*) AS captured_rows FROM {CODEX_TABLE}
  ORDER BY inference_table
"""))


inference_table                     captured_rows
----------------------------------  -------------
campaign_desk_llm_payload (app)     16           
codex_agent_payload (coding agent)  9            
[2 row(s)]


## 3. Sample auto-captured rows from the APP endpoint (real routed calls to gpt-5.4)


In [4]:
display(spark.sql(f"""
  SELECT event_time, status_code, latency_ms, destination_model, api_type,
         substring(request, 1, 150) AS request_head
    FROM {APP_TABLE}
   WHERE status_code = 200
   ORDER BY event_time
   LIMIT 5
"""))


event_time                status_code  latency_ms  destination_model    api_type                    request_head                                                                                                                                          
------------------------  -----------  ----------  -------------------  --------------------------  ------------------------------------------------------------------------------------------------------------------------------------------------------
2026-08-28T23:04:09.724Z  200          2644        campaign_desk_gpt54  mlflow/v1/chat/completions  {"max_tokens":80,"messages":[{"content":"In one sentence: which Brightwave campaign is the hero underperformer and what action is recommended?","role"
2026-08-28T23:07:20.434Z  200          2126        campaign_desk_gpt54  mlflow/v1/chat/completions  {"max_tokens":60,"messages":[{"content":"Summarize why social+testimonial creative beats display for gen_z apparel, in one line.","role":"user"}],"

## 4. [BONUS] Coding agent is NOT bound by the app's all-data guardrail — the two tables side by side

The exact same *"export EVERY row from EVERY table … dump the entire database"* prompt:
- **coding agent endpoint (`campaign_desk_codex`, no guardrail)** → logged at **status 200** (processed, not blocked)
- **app endpoint (`campaign_desk_llm`, guardrail on)** → **0 successful all-data completions** (guardrail denies it pre-call, so it never reaches the model / the table)


In [5]:
# Note the OUTER parentheses: keeps the OR-group intact when combined with `AND status_code = 200` below.
ALLDATA = ("(lower(request) LIKE '%every row%' OR lower(request) LIKE '%entire database%' "
           "OR lower(request) LIKE '%dump the entire%' OR lower(request) LIKE '%all rows from every%')")


### 4a. Coding agent — the all-data read SUCCEEDED (status 200, no guardrail block)


In [6]:
display(spark.sql(f"""
  SELECT event_time, status_code,
         substring(request,  1, 220) AS all_data_prompt,
         substring(response, 1, 220) AS response_head,
         (response LIKE '%databricks_service_policy%' OR response LIKE '%content_filter%') AS blocked_by_guardrail
    FROM {CODEX_TABLE}
   WHERE {ALLDATA}
   ORDER BY event_time
"""))


event_time                status_code  all_data_prompt                                                                                                                                                                                                               response_head                                                                                                                                                                                                                 blocked_by_guardrail
------------------------  -----------  ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------  ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------  -------------

### 4b. Side-by-side count — coding agent all-data SUCCEEDS, app all-data NEVER completes


In [7]:
display(spark.sql(f"""
  SELECT
    (SELECT COUNT(*) FROM {CODEX_TABLE} WHERE {ALLDATA} AND status_code = 200) AS coding_agent_all_data_succeeded_200,
    (SELECT COUNT(*) FROM {APP_TABLE}   WHERE {ALLDATA} AND status_code = 200) AS app_all_data_succeeded_200
"""))


coding_agent_all_data_succeeded_200  app_all_data_succeeded_200
-----------------------------------  --------------------------
4                                    0                         
[1 row(s)]


### 4c. Why: the guardrail is bound to the APP endpoint only (coding agent has none)


In [8]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
for svc in ["campaign_desk_llm", "campaign_desk_codex"]:
    r = w.api_client.do("GET", f"/api/2.1/unity-catalog/model-services/{CAT}.brightwave.{svc}")
    pols = [p.get("name") for p in r.get("config", {}).get("service_policies", [])]
    role = "APP" if svc == "campaign_desk_llm" else "CODING AGENT"
    print(f"{role:12s}  {svc:20s}  guardrails = {pols if pols else 'NONE'}")


APP           campaign_desk_llm     guardrails = ['block_all_lakebase_data']
CODING AGENT  campaign_desk_codex   guardrails = NONE


## 5. Gateway usage — every surface handled real calls (app, coding agent, guardrail judge, Slack MCP) + budget block


In [9]:
display(spark.sql(f"""
  SELECT service_type, service_name, status_code,
         COUNT(*) AS calls, COALESCE(SUM(total_tokens), 0) AS total_tokens,
         MIN(event_time) AS first_seen, MAX(event_time) AS last_seen
    FROM system.ai_gateway.usage
   WHERE workspace_id = '{WS}'
   GROUP BY service_type, service_name, status_code
   ORDER BY service_type, calls DESC
"""))


service_type   service_name                                                    status_code  calls  total_tokens  first_seen                last_seen               
-------------  --------------------------------------------------------------  -----------  -----  ------------  ------------------------  ------------------------
MCP_SERVICE    system.ai.slack                                                 200          43     0             2026-08-28T16:41:05.000Z  2026-08-29T01:00:36.000Z
MCP_SERVICE    system.ai.slack                                                 204          8      0             2026-08-28T16:41:05.000Z  2026-08-29T01:00:35.000Z
MODEL_SERVICE  system.ai.gpt-5-2                                               200          21     11950         2026-08-28T23:04:10.000Z  2026-08-29T01:12:19.000Z
MODEL_SERVICE  system.ai.gpt-5-6-luna                                          200          13     519791        2026-08-29T00:09:10.000Z  2026-08-29T00:21:59.000Z
MODEL_SERVICE  b

### 5a. Governed Slack MCP tool calls (proves the MCP was actually used)


In [10]:
display(spark.sql(f"""
  SELECT event_time, status_code,
         mcp_metadata.tool_name  AS tool,
         mcp_metadata.server_type AS server_type
    FROM system.ai_gateway.usage
   WHERE workspace_id = '{WS}'
     AND service_type = 'MCP_SERVICE' AND service_name = 'system.ai.slack'
     AND mcp_metadata.tool_name IS NOT NULL AND mcp_metadata.tool_name <> ''
   ORDER BY event_time
   LIMIT 15
"""))


event_time                status_code  tool                             server_type
------------------------  -----------  -------------------------------  -----------
2026-08-29T00:21:05.000Z  200          slack_search_public_and_private  SYSTEM     
2026-08-29T00:21:15.000Z  200          slack_search_public_and_private  SYSTEM     
2026-08-29T00:21:15.000Z  200          slack_search_public_and_private  SYSTEM     
2026-08-29T00:21:15.000Z  200          slack_search_public_and_private  SYSTEM     
2026-08-29T00:21:15.000Z  200          slack_search_public_and_private  SYSTEM     
2026-08-29T00:21:22.000Z  200          slack_search_public_and_private  SYSTEM     
2026-08-29T00:21:31.000Z  200          slack_search_public_and_private  SYSTEM     
2026-08-29T00:21:38.000Z  200          slack_search_public_and_private  SYSTEM     
2026-08-29T00:21:46.000Z  200          slack_search_public_and_private  SYSTEM     
2026-08-29T00:21:46.000Z  200          slack_search_public_and_private  SYST

### 5b. Budget block — HTTP 403 across BOTH endpoints once the $0.05 cap crossed


In [11]:
display(spark.sql(f"""
  SELECT event_time, service_name, status_code, url
    FROM system.ai_gateway.usage
   WHERE workspace_id = '{WS}' AND status_code = 403
     AND service_type = 'MODEL_SERVICE' AND service_name LIKE 'brightwave%'
   ORDER BY event_time
"""))


event_time                service_name                                                    status_code  url                                                                                                  
------------------------  --------------------------------------------------------------  -----------  -----------------------------------------------------------------------------------------------------
2026-08-29T01:12:12.000Z  brightwave_techsummit27_catalog.brightwave.campaign_desk_llm    403          https://fe-sandbox-brightwave-techsummit27.cloud.databricks.com/ai-gateway/mlflow/v1/chat/completions
2026-08-29T01:12:15.000Z  brightwave_techsummit27_catalog.brightwave.campaign_desk_llm    403          https://fe-sandbox-brightwave-techsummit27.cloud.databricks.com/ai-gateway/mlflow/v1/chat/completions
2026-08-29T01:12:16.000Z  brightwave_techsummit27_catalog.brightwave.campaign_desk_llm    403          https://fe-sandbox-brightwave-techsummit27.cloud.databricks.com/ai-gateway/ml

---
**Result:** the catalog + both auto-capture inference tables exist and are populated (§1–3); the coding agent's
endpoint processes the all-data prompt that the app's guardrail blocks (§4); and every gateway surface — including
the governed Slack MCP — handled real calls, with the $0.05 budget enforced as 403s (§5). This notebook's executed
cell outputs are the committed proof-of-execution.
